# Fase 1.2 — Limpieza inicial y carga tipada de listings

Proyecto: análisis de la oferta y estimación de precios de Airbnb en Río de Janeiro.

## Punto de partida comprobado

Tus informes muestran 48 247 registros, 90 columnas, 48 247 IDs distintos y ninguna fila completamente duplicada. CSV y PostgreSQL coinciden en encabezados, conteos e IDs. Hay 4 100 precios ausentes (8,50 %) y 44 147 precios positivos interpretables. Hay 14 columnas totalmente vacías. `bathrooms` está vacía y `bathrooms_text` tiene solo 59 ausencias.

## Resultado de este notebook

- Crea una **selección inicial de columnas tipadas** en `clean.listings` dentro de `proy_airbnb_rio`.
- Conserva todos los anuncios, incluidos los que no tienen precio. Los marca con indicadores de calidad.
- Conserva dormitorios, camas, noches y precios extremos; los revisaremos en el EDA.
- No imputa valores ni entrena modelos. No altera `staging.listings`.
- Produce informes de transformaciones, reglas, formatos pendientes y perfil de precios en `reports/limpieza/`.

**Ejecución:** guarda este archivo en `notebooks/`, selecciona tu kernel `.venv` y ejecuta las 9 celdas de código en orden. La **celda 8 escribe en PostgreSQL** y falla si `clean.listings` ya existe; no la ejecutes para reemplazar una tabla. Las celdas anteriores trabajan en memoria y guardan informes locales. Detente ante errores de ejecución. Las alertas de revisión son resultados que analizaremos.

No necesitas volver a importar listings ni instalar librerías nuevas. La única contraseña solicitada se mantiene en memoria mientras está abierto el kernel; no la escribas como texto ni la muestres en salidas.

## 1. Preparar el entorno y las rutas

No es necesario copiar aquí las variables ni el DataFrame del notebook anterior: este notebook es independiente.

In [1]:
import sys
import re
import json
from decimal import Decimal, InvalidOperation
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine import URL
from sqlalchemy.types import Text, BigInteger, Float, Numeric, Boolean, Date
from IPython.display import display

HOST, PORT = 'localhost', 5432
DATABASE, USER = 'proy_airbnb_rio', 'postgres'
RAIZ_MANUAL = ''  # Si es necesario, escribe aquí la carpeta principal del proyecto.
if RAIZ_MANUAL:
    RAIZ = Path(RAIZ_MANUAL).expanduser().resolve()
else:
    inicio = Path.cwd().resolve()
    RAIZ = next((p for p in [inicio, *inicio.parents]
                 if (p / 'data' / 'raw').is_dir() and (p / 'src').is_dir()), None)
if RAIZ is None or not (RAIZ / 'data' / 'raw').is_dir():
    raise RuntimeError('Configura RAIZ_MANUAL con la carpeta principal del proyecto.')
REPORTES = RAIZ / 'reports' / 'limpieza'

# Convención documentada: price es un precio diario en moneda local.
# Para el mercado de Río de Janeiro se etiqueta BRL; no se convierte el importe.
# El signo $ del CSV no se interpreta como USD.
MONEDA_PRECIO = 'BRL'
FUENTE_MONEDA = 'Inside Airbnb Data Dictionary v4.3: price, moneda local; mercado Brasil'
print('Python:', sys.executable)
print('Proyecto:', RAIZ)
print('Destino previsto:', DATABASE, '→ clean.listings')

Python: c:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\.venv\Scripts\python.exe
Proyecto: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model
Destino previsto: proy_airbnb_rio → clean.listings


## 2. Leer staging y validar la identidad de los registros

La lectura usa una transacción de solo lectura. El motor se reutiliza en las últimas celdas y se cierra al terminar. Si cierras el kernel, vuelve a ejecutar desde la celda 1.

In [2]:
motor = create_engine(
    URL.create('postgresql+psycopg2', username=USER,
               password=getpass(f'Contraseña de {USER}: '),
               host=HOST, port=PORT, database=DATABASE),
    connect_args={'connect_timeout': 10}, pool_pre_ping=True,
)
try:
    with motor.connect() as conexion:
        with conexion.begin():
            conexion.execute(text('SET TRANSACTION ISOLATION LEVEL REPEATABLE READ, READ ONLY'))
            original = pd.read_sql_query(text('SELECT * FROM staging.listings'), conexion)
except Exception:
    motor.dispose()
    raise

requeridas = ['id', 'host_id', 'scrape_id', 'last_scraped', 'neighbourhood_cleansed',
              'room_type', 'property_type', 'latitude', 'longitude', 'accommodates',
              'price', 'bathrooms_text', 'number_of_reviews']
faltantes = sorted(set(requeridas) - set(original.columns))
if faltantes:
    raise ValueError(f'Faltan columnas necesarias: {faltantes}')
ids_control = original['id'].astype('string').str.strip()
if original.empty or ids_control.isna().any() or ids_control.eq('').any() or ids_control.duplicated().any():
    raise ValueError('La carga está vacía o tiene IDs ausentes/repetidos. Revisar antes de transformar.')
if not ids_control.str.fullmatch(r'[0-9]+').all():
    raise ValueError('Hay identificadores con formato distinto de una cadena de dígitos.')
print(f'Leídos {len(original):,} registros y {original.shape[1]} columnas.')
if (len(original), original.shape[1]) != (48247, 90):
    print('AVISO: este tamaño difiere del diagnóstico recibido; revisa si cambió el corte.')

Leídos 48,247 registros y 90 columnas.


## 3. Funciones de normalización y registro de incidencias

Solo se normalizan ausencias reconocidas en las columnas seleccionadas. Los IDs se mantienen como texto. No se aplican escalado, codificación de categorías ni imputaciones que deban aprenderse del conjunto de entrenamiento.

In [4]:
incidencias_partes = []
auditoria = []
tipos_sql = {}
limpio = pd.DataFrame(index=original.index)

def texto_limpio(s):
    resultado = s.astype('string').str.strip()
    marcadores = resultado.str.lower().isin(['', 'null', 'none', 'nan', 'n/a', 'na'])
    return resultado.mask(marcadores)

def registrar(columna, regla, mascara):
    mascara = pd.Series(mascara, index=original.index).fillna(False).astype(bool)
    n = int(mascara.sum())
    auditoria.append({'columna': columna, 'regla': regla, 'registros': n})
    if n:
        incidencias_partes.append(pd.DataFrame({
            'id': ids_control.loc[mascara].to_numpy(),
            'columna': columna, 'regla': regla,
            'valor_original': original.loc[mascara, columna].astype('string').to_numpy(),
        }))

def agregar_numero(c, entero=False):
    s = texto_limpio(original[c])
    numero = pd.to_numeric(s, errors='coerce').astype('Float64')
    finito = numero.notna() & np.isfinite(numero.fillna(0).astype(float))
    valido = finito.copy()
    if entero:
        # Evitar redondear fracciones o exceder el rango entero seguro de esta conversión.
        valido = valido & numero.mod(1).eq(0).fillna(False) & numero.abs().le(2**53).fillna(False)
    registrar(c, 'formato_numerico_no_admitido', s.notna() & ~valido)
    limpio[c] = numero.where(valido).astype('Int64' if entero else 'Float64')
    tipos_sql[c] = BigInteger() if entero else Float()

PATRON_PRECIO = re.compile(r'(?:R\$\s*|\$\s*)?-?(?:[0-9]+|[0-9]{1,3}(?:,[0-9]{3})+)(?:\.[0-9]{1,2})?')
def convertir_precio(valor):
    if pd.isna(valor):
        return None
    s = str(valor).strip()
    if not PATRON_PRECIO.fullmatch(s):
        return None
    s = re.sub(r'^(?:R\$|\$)\s*', '', s).replace(',', '')
    try:
        n = Decimal(s)
        return n if n.is_finite() else None
    except InvalidOperation:
        return None

## 4. Construir la tabla de trabajo con tipos explícitos

Esta selección conserva variables útiles para el mercado, geografía, valoraciones y características del anuncio. Los campos no seleccionados siguen disponibles en staging. Las columnas `price_quote_*`, los ingresos y la ocupación estimados no se incluyen como predictores de precios. Tampoco se usarán como predictores `price_missing` ni `price_positive`, que dependen del precio objetivo.

Se conserva `amenities` como texto original normalizado para procesarlo más adelante. La moneda BRL se documenta a partir del diccionario público de la fuente; no se realiza conversión monetaria.

In [5]:
columnas_texto = ['id', 'host_id', 'scrape_id', 'neighbourhood_cleansed',
                  'room_type', 'property_type', 'bathrooms_text', 'amenities']
columnas_enteras = ['accommodates', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights',
                    'host_listings_count', 'calculated_host_listings_count',
                    'calculated_host_listings_count_entire_homes',
                    'calculated_host_listings_count_private_rooms',
                    'calculated_host_listings_count_shared_rooms',
                    'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d',
                    'availability_30', 'availability_60', 'availability_90', 'availability_365']
columnas_decimales = ['latitude', 'longitude', 'reviews_per_month', 'review_scores_rating',
                      'review_scores_accuracy', 'review_scores_cleanliness',
                      'review_scores_checkin', 'review_scores_communication',
                      'review_scores_location', 'review_scores_value']
columnas_fechas = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']
columnas_booleanas = ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']

for c in columnas_texto:
    if c in original:
        limpio[c] = texto_limpio(original[c])
        tipos_sql[c] = Text()
for c in columnas_enteras:
    if c in original:
        agregar_numero(c, entero=True)
for c in columnas_decimales:
    if c in original:
        agregar_numero(c)
for c in columnas_fechas:
    if c not in original:
        continue
    s = texto_limpio(original[c])
    fecha = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
    registrar(c, 'fecha_no_interpretable', s.notna() & fecha.isna())
    limpio[c] = fecha.dt.date.where(fecha.notna(), None)
    tipos_sql[c] = Date()
equivalencias = {'t': True, 'true': True, 'f': False, 'false': False}
for c in columnas_booleanas:
    if c not in original:
        continue
    s = texto_limpio(original[c]).str.lower()
    registrar(c, 'booleano_no_reconocido', s.notna() & ~s.isin(equivalencias))
    limpio[c] = s.map(equivalencias).astype('boolean')
    tipos_sql[c] = Boolean()

precio_origen = texto_limpio(original['price'])
limpio['price'] = precio_origen.map(convertir_precio).astype(object)
registrar('price', 'precio_no_interpretable', precio_origen.notna() & limpio['price'].isna())
limpio['price_currency'] = MONEDA_PRECIO
tipos_sql.update({'price': Numeric(), 'price_currency': Text()})
print(f'Tabla de trabajo inicial: {limpio.shape[0]:,} filas y {limpio.shape[1]} columnas.')
display(limpio[['id', 'neighbourhood_cleansed', 'accommodates', 'price', 'price_currency']].head())

Tabla de trabajo inicial: 48,247 filas y 45 columnas.


,id,neighbourhood_cleansed,accommodates,price,price_currency
0,17878,Copacabana,5,513.00,BRL
1,25026,Copacabana,3,422.00,BRL
2,35764,Copacabana,2,366.00,BRL
3,48305,Ipanema,13,3761.43,BRL
4,48901,Copacabana,9,1173.25,BRL


## 5. Extraer baños y añadir indicadores de revisión

`bathrooms_text` se conserva. La derivación reconoce formatos ingleses explícitos, como `1 bath`, `1.5 baths`, `1 shared bath`, `2 private baths` y `Half-bath`. Los formatos no reconocidos quedan con baños derivados ausentes y se reportan; nunca se convierten en cero.

`shared=False` solo se asigna cuando el texto dice `private`; si no especifica el uso, queda desconocido. Las coordenadas solo se revisan contra límites globales, todavía no contra polígonos de Río. Un anuncio con muchas camas o dormitorios se marca para revisión, no se descarta.

In [6]:
def extraer_banos(valor):
    if pd.isna(valor):
        return (None, None)
    s = str(valor).strip().lower()
    palabras = re.findall(r'[a-z]+', s)
    compartido = True if 'shared' in palabras else (False if 'private' in palabras else None)
    if re.fullmatch(r'(?:(?:shared|private)[ ]+)?half[- ]bath', s):
        return (0.5, compartido)
    m = re.fullmatch(r'([0-9]+(?:[.][0-9]+)?)[ ]+(?:(?:shared|private)[ ]+)?(?:baths?|bathrooms?)', s)
    return (float(m.group(1)), compartido) if m else (None, compartido)

banos = limpio['bathrooms_text'].map(extraer_banos)
limpio['bathrooms_derived'] = pd.Series([v[0] for v in banos], index=limpio.index, dtype='Float64')
limpio['bathroom_shared'] = pd.Series([v[1] for v in banos], index=limpio.index, dtype='boolean')
tipos_sql.update({'bathrooms_derived': Float(), 'bathroom_shared': Boolean()})
registrar('bathrooms_text', 'banos_texto_no_reconocido',
          limpio['bathrooms_text'].notna() & limpio['bathrooms_derived'].isna())

limpio['price_missing'] = limpio['price'].isna()
limpio['price_positive'] = limpio['price'].map(lambda x: x is not None and pd.notna(x) and x > 0).astype(bool)
limpio['geo_needs_review'] = ~(
    limpio['latitude'].between(-90, 90) & limpio['longitude'].between(-180, 180)
).fillna(False)
limpio['capacity_needs_review'] = ~limpio['accommodates'].ge(1).fillna(False)
for c in ['beds', 'bedrooms']:
    if c in limpio:
        limpio['capacity_needs_review'] |= limpio[c].lt(0).fillna(False)
        limpio['capacity_needs_review'] |= limpio[c].gt(limpio['accommodates']).fillna(False)
limpio['nights_needs_review'] = False
if {'minimum_nights', 'maximum_nights'} <= set(limpio.columns):
    minimo, maximo = limpio['minimum_nights'], limpio['maximum_nights']
    limpio['nights_needs_review'] = (minimo.isna() | maximo.isna() |
        minimo.lt(1).fillna(False) | maximo.lt(1).fillna(False) | minimo.gt(maximo).fillna(False))

for c in ['price_missing', 'price_positive', 'geo_needs_review', 'capacity_needs_review', 'nights_needs_review']:
    limpio[c] = limpio[c].astype('boolean')
    tipos_sql[c] = Boolean()

perfil_banos = limpio.groupby(['bathrooms_text', 'bathrooms_derived', 'bathroom_shared'], dropna=False).size().reset_index(name='anuncios')
display(perfil_banos.sort_values('anuncios', ascending=False).head(20))
print('Registros marcados para revisar características:', int(limpio['capacity_needs_review'].sum()))
print('Estas marcas no son un filtro definitivo para el modelo ni para el dashboard.')

,bathrooms_text,bathrooms_derived,bathroom_shared,anuncios
2,1 bath,1.0,<NA>,26052
17,2 baths,2.0,<NA>,8179
3,1 private bath,1.0,False,3308
4,1 shared bath,1.0,True,2629
5,1.5 baths,1.5,<NA>,1907
21,3 baths,3.0,<NA>,1635
19,2.5 baths,2.5,<NA>,1049
18,2 shared baths,2.0,True,608
25,4 baths,4.0,<NA>,381
23,3.5 baths,3.5,<NA>,359


Registros marcados para revisar características: 1411
Estas marcas no son un filtro definitivo para el modelo ni para el dashboard.


## 6. Controles de conservación, calidad y cobertura

Los valores imposibles por formato pasan a NULL en la capa tipada y quedan registrados con su ID; los originales permanecen en staging. Las alertas semánticas se conservan sin borrar filas. Para estudiar oferta se usan todos los anuncios; para precios se filtra `price_positive=True`. Este filtro todavía no define el conjunto final de ML.

Se comprueba también la relación entre ausencia de valoración y anuncios sin reseñas; no se supone esa relación a partir de los conteos separados.

In [7]:
if len(limpio) != len(original) or not limpio['id'].equals(ids_control):
    raise RuntimeError('Se alteró la cantidad o identidad de los anuncios.')
if limpio['id'].isna().any() or limpio['id'].duplicated().any():
    raise RuntimeError('Los identificadores de la capa tipada no son válidos.')
if set(tipos_sql) != set(limpio.columns):
    raise RuntimeError('Hay columnas sin tipo SQL explícito.')

auditoria_df = pd.DataFrame(auditoria)
incidencias_df = (pd.concat(incidencias_partes, ignore_index=True) if incidencias_partes
                  else pd.DataFrame(columns=['id', 'columna', 'regla', 'valor_original']))
vacias_origen = [c for c in original if texto_limpio(original[c]).isna().all()]
precio_float = pd.Series([float(x) if pd.notna(x) else np.nan for x in limpio['price']], index=limpio.index)
perfil_precio = precio_float[limpio['price_positive'].astype(bool)].describe(
    percentiles=[.01, .25, .5, .75, .95, .99]
).rename_axis('estadistico').reset_index(name='price')
perfil_tipos = pd.DataFrame([{
    'columna': c, 'tipo_pandas': str(limpio[c].dtype),
    'tipo_sql': str(tipos_sql[c]), 'ausentes': int(limpio[c].isna().sum())
} for c in limpio])
resumen_limpieza = {
    'generado_utc': datetime.now(timezone.utc).isoformat(),
    'origen': 'staging.listings', 'destino': 'clean.listings',
    'filas_origen': len(original), 'filas_preparadas': len(limpio),
    'filas_eliminadas': 0, 'columnas_preparadas': len(limpio.columns),
    'ids_unicos': int(limpio['id'].nunique()),
    'precio_ausente_despues_tipado': int(limpio['price_missing'].sum()),
    'precio_positivo': int(limpio['price_positive'].sum()),
    'precio_no_interpretable': int(auditoria_df.loc[auditoria_df['columna'].eq('price'), 'registros'].sum()),
    'precio_no_positivo_no_nulo': int((limpio['price'].notna() & ~limpio['price_positive']).sum()),
    'moneda': MONEDA_PRECIO, 'fundamento_moneda': FUENTE_MONEDA,
    'columnas_sin_informacion_en_origen': vacias_origen,
    'banos_ausentes_en_texto': int(limpio['bathrooms_text'].isna().sum()),
    'banos_derivados_ausentes': int(limpio['bathrooms_derived'].isna().sum()),
    'incidencias_de_formato_registradas': len(incidencias_df),
    'geografia_por_revisar': int(limpio['geo_needs_review'].sum()),
    'capacidad_por_revisar': int(limpio['capacity_needs_review'].sum()),
    'noches_por_revisar': int(limpio['nights_needs_review'].sum()),
    'fechas_raspado_distintas': sorted(original['last_scraped'].dropna().astype(str).unique().tolist()),
    'carga_postgresql': 'pendiente',
}
if {'review_scores_rating', 'number_of_reviews'} <= set(limpio.columns):
    sin_valoracion = limpio['review_scores_rating'].isna()
    resumen_limpieza['sin_valoracion_y_cero_resenas'] = int((sin_valoracion & limpio['number_of_reviews'].eq(0)).sum())
    resumen_limpieza['sin_valoracion_y_con_resenas'] = int((sin_valoracion & limpio['number_of_reviews'].gt(0)).sum())
display(pd.Series(resumen_limpieza).to_frame('resultado'))
display(auditoria_df.loc[auditoria_df['registros'].gt(0)])
display(perfil_precio)
if (len(original), original.shape[1]) == (48247, 90):
    if resumen_limpieza['precio_positivo'] != 44147 or resumen_limpieza['precio_ausente_despues_tipado'] != 4100:
        raise RuntimeError('La transformación del precio difiere del diagnóstico. Revisar antes de cargar.')

,resultado
generado_utc,2026-11-21T19:07:11.403042+00:00
origen,staging.listings
destino,clean.listings
filas_origen,48247
filas_preparadas,48247
filas_eliminadas,0
columnas_preparadas,52
ids_unicos,48247
precio_ausente_despues_tipado,4100
precio_positivo,44147


,columna,regla,registros
36,bathrooms_text,banos_texto_no_reconocido,1


,estadistico,price
0,count,44147.000000
1,mean,860.699080
2,std,4434.603391
3,min,18.970000
4,1%,90.000000
5,25%,300.000000
6,50%,450.000000
7,75%,750.000000
8,95%,2175.000000
9,99%,7553.774600


## 7. Guardar informes y reglas aplicadas

`incidencias_formato.csv` incluye solo ID, columna, regla y valor problemático de las variables técnicas seleccionadas. No se exportan nombres, perfiles ni descripciones de anfitriones. Los informes de esta etapa se reemplazan si vuelves a ejecutar esta celda.

In [8]:
REPORTES.mkdir(parents=True, exist_ok=True)
reglas = [
    ['conservacion', 'Conservar todos los IDs; no eliminar anuncios sin precio.'],
    ['identificadores', 'Normalizar espacios y almacenar IDs como TEXT.'],
    ['precio', 'Interpretar símbolo opcional y formato con punto decimal/coma de miles; conservar NULL.'],
    ['moneda', 'BRL por la definición de precio en moneda local del diccionario; sin conversión.'],
    ['numeros_fechas', 'Tipar datos; formato no reconocido a NULL y registro de incidencia.'],
    ['banos', 'Derivar solo formatos reconocidos; conservar bathrooms_text; desconocido permanece NULL.'],
    ['extremos', 'No eliminar ni recortar precios, camas, dormitorios o noches por percentiles.'],
    ['imputacion', 'No imputar precios ni aprender estadísticas para ML en esta etapa.'],
    ['predictores', 'Excluir price_quote_*, ingresos/ocupación estimados y flags derivados de price de los predictores.'],
    ['disponibilidad', 'Conservar días disponibles; no interpretarlos como ocupación observada.'],
]
pd.DataFrame(reglas, columns=['tema', 'regla']).to_csv(REPORTES / 'reglas_limpieza.csv', index=False, encoding='utf-8-sig')
for nombre, tabla in {
    'auditoria_transformaciones.csv': auditoria_df,
    'incidencias_formato.csv': incidencias_df,
    'perfil_banos.csv': perfil_banos,
    'perfil_tipos_limpios.csv': perfil_tipos,
    'perfil_precio_limpio.csv': perfil_precio,
}.items():
    tabla.to_csv(REPORTES / nombre, index=False, encoding='utf-8-sig')

def guardar_resumen():
    (REPORTES / 'resumen_limpieza.json').write_text(
        json.dumps(resumen_limpieza, ensure_ascii=False, indent=2), encoding='utf-8')
guardar_resumen()
print('Informes guardados en:', REPORTES)
print('La carga de PostgreSQL sigue pendiente: corresponde a la siguiente celda.')

Informes guardados en: C:\Users\user\Documents\Inteligencia de negocios\Proyecto_IN\Proy_Analysis_Airbnb_Business_Model\reports\limpieza
La carga de PostgreSQL sigue pendiente: corresponde a la siguiente celda.


## 8. Crear y cargar `clean.listings` en PostgreSQL

**Esta celda escribe en la base de datos.** Crea el esquema `clean`, la tabla tipada y su clave primaria. Usa una transacción: si falla la inserción o la validación, se revierte la operación. Si `clean.listings` ya existe, se detiene sin borrarla ni añadir filas.

Se conservarán las 48 247 filas si sigues trabajando con el mismo corte. Un precio NULL no impide cargar un anuncio. Ejecuta esta celda una vez cuando las celdas previas terminen correctamente.

In [9]:
try:
    with motor.begin() as conexion:
        if inspect(conexion).has_table('listings', schema='clean'):
            raise RuntimeError('clean.listings ya existe. No se borrará ni duplicará. Si ya la cargaste con este notebook, verifica con la celda 9.')
        conexion.execute(text('CREATE SCHEMA IF NOT EXISTS clean'))
        limpio.to_sql('listings', con=conexion, schema='clean', if_exists='fail',
                      index=False, dtype=tipos_sql, chunksize=500, method='multi')
        conexion.execute(text('ALTER TABLE clean.listings ADD PRIMARY KEY (id)'))
        cargadas = int(conexion.execute(text('SELECT COUNT(*) FROM clean.listings')).scalar_one())
        if cargadas != len(limpio):
            raise RuntimeError('El conteo de destino no coincide: se revierte la carga.')
    resumen_limpieza['carga_postgresql'] = 'completada'
    resumen_limpieza['filas_cargadas'] = cargadas
    guardar_resumen()
    print(f'Carga confirmada: {cargadas:,} filas en {DATABASE} → clean.listings.')
except Exception:
    motor.dispose()
    raise

Carga confirmada: 48,247 filas en proy_airbnb_rio → clean.listings.


## 9. Verificar la carga y cerrar la conexión

Actualiza el árbol de pgAdmin para ver `Schemas → clean → Tables → listings`. Puedes ejecutar el SELECT de esta celda en Query Tool. Los tipos ya serán numéricos, fechas y booleanos según corresponda.

Si la celda 8 se detuvo porque la tabla ya existía, esta celda permite revisar los conteos existentes, pero no certifica por sí sola que todos sus campos coincidan con la versión en memoria.

In [10]:
try:
    with motor.connect() as conexion:
        with conexion.begin():
            conexion.execute(text('SET TRANSACTION READ ONLY'))
            verificacion = pd.read_sql_query(text("""
                SELECT COUNT(*) AS anuncios,
                       COUNT(DISTINCT id) AS ids_distintos,
                       COUNT(*) FILTER (WHERE price IS NULL) AS anuncios_sin_precio,
                       COUNT(*) FILTER (WHERE price > 0) AS anuncios_con_precio_positivo
                FROM clean.listings
            """), conexion)
    display(verificacion)
finally:
    motor.dispose()
print('Conexiones cerradas. Revisa los informes de limpieza antes del EDA.')

,anuncios,ids_distintos,anuncios_sin_precio,anuncios_con_precio_positivo
0,48247,48247,4100,44147


Conexiones cerradas. Revisa los informes de limpieza antes del EDA.


## Qué compartir al terminar

Comparte `resumen_limpieza.json`, `auditoria_transformaciones.csv`, `perfil_banos.csv` y `perfil_precio_limpio.csv`. Con ellos revisaremos los formatos de baños, valores extremos y las alertas antes de desarrollar el EDA geográfico.

La capa `clean` es una preparación inicial para análisis; todavía no es el modelo dimensional de Power BI ni la matriz final de entrenamiento. Los valores faltantes de variables predictoras se tratarán dentro de los pipelines después de separar entrenamiento y prueba.

Documentación: [diccionario de Inside Airbnb (versión pública v4.3)](https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit), [pandas.to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html).

**Alcance de la revisión del archivo entregado:** sintaxis y transformaciones probadas con datos sintéticos. La conexión y la carga a tu PostgreSQL local se verificarán cuando lo ejecutes. No se ha ejecutado la escritura en tu equipo desde esta conversación.